# RSNA Knee Abnormality Detection - baseline

Notebook lista para ejecutarse en Kaggle sin Internet. Agrega la competencia como fuente de datos y activa GPU si est? disponible. El entregable final ser? `/kaggle/working/submission.csv`.

El baseline combina TF-IDF del reporte con estad?sticas de intensidad DICOM y metadatos de series. Es reproducible y CPU-friendly; no es una validaci?n cl?nica.

In [ ]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import pydicom
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
ID = "StudyInstanceUID"
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
print("Configuration ready")

In [ ]:
roots = [Path("/kaggle/input/rsna-knee-abnormality-detection"), Path("/kaggle/input/rsna-knee-abnormalities-detection")]
DATA_ROOT = next((p for p in roots if p.exists()), None)
if DATA_ROOT is None:
    found = list(Path("/kaggle/input").glob("**/train.csv"))
    DATA_ROOT = found[0].parent if found else None
if DATA_ROOT is None:
    raise FileNotFoundError("Attach the RSNA competition dataset to this notebook")
TRAIN_CSV, TEST_CSV = DATA_ROOT / "train.csv", DATA_ROOT / "test.csv"
TRAIN_SERIES_CSV, TEST_SERIES_CSV = DATA_ROOT / "train_series.csv", DATA_ROOT / "test_series.csv"
TRAIN_ROOT, TEST_ROOT = DATA_ROOT / "train_series", DATA_ROOT / "test_series"
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train_series = pd.read_csv(TRAIN_SERIES_CSV) if TRAIN_SERIES_CSV.exists() else pd.DataFrame()
test_series = pd.read_csv(TEST_SERIES_CSV) if TEST_SERIES_CSV.exists() else pd.DataFrame()
missing = [c for c in TARGETS if c not in train.columns]
if missing:
    raise ValueError(f"Missing targets: {missing}")
print(DATA_ROOT)
print("train", train.shape, "test", test.shape, "series", train_series.shape)

## Feature extraction

Se muestrean hasta 24 slices por estudio para mantener el tiempo de ejecuci?n. Se calculan cuantiles de intensidad, cantidad de slices y descriptores de `train_series.csv`. En test el reporte est? vac?o, pero el mismo flujo funciona sin texto.

In [ ]:
def series_features(study_id, frame):
    if frame.empty:
        return np.zeros(6, dtype=np.float32)
    rows = frame[frame[ID].astype(str) == str(study_id)]
    return np.array([
        len(rows),
        rows["Fluid_Sensitive"].sum() if "Fluid_Sensitive" in rows else 0,
        rows["Fat_Suppression"].sum() if "Fat_Suppression" in rows else 0,
        (rows["Anatomical_Plane"] == "Sagittal").sum() if "Anatomical_Plane" in rows else 0,
        (rows["Anatomical_Plane"] == "Coronal").sum() if "Anatomical_Plane" in rows else 0,
        (rows["Anatomical_Plane"] == "Axial").sum() if "Anatomical_Plane" in rows else 0,
    ], dtype=np.float32)

def image_features(study_root, max_slices=24):
    paths = sorted(p for p in Path(study_root).rglob("*") if p.is_file())
    if len(paths) > max_slices:
        paths = [paths[i] for i in np.linspace(0, len(paths) - 1, max_slices, dtype=int)]
    samples = []
    for path in paths:
        try:
            pixels = np.asarray(pydicom.dcmread(path, force=True).pixel_array, dtype=np.float32)
            if pixels.size:
                pixels = np.nan_to_num(pixels)
                lo, hi = np.percentile(pixels, [1, 99])
                if hi > lo:
                    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
                samples.append(pixels.ravel())
        except Exception:
            continue
    if not samples:
        return np.zeros(10, dtype=np.float32)
    values = np.concatenate(samples)
    return np.array([values.mean(), values.std(), *np.percentile(values, [1, 10, 25, 50, 75, 90, 99]), len(samples) / max_slices], dtype=np.float32)

def build_features(frame, image_root, series_frame, has_report):
    texts, numeric = [], []
    for n, (_, row) in enumerate(frame.iterrows(), 1):
        study_id = str(row[ID])
        texts.append(str(row.get("Report", "")) if has_report and pd.notna(row.get("Report", "")) else "")
        numeric.append(np.concatenate([image_features(Path(image_root) / study_id), series_features(study_id, series_frame)]))
        if n % 100 == 0:
            print(f"Processed {n}/{len(frame)}")
    return texts, np.vstack(numeric)

train_text, train_numeric = build_features(train, TRAIN_ROOT, train_series, True)
test_text, test_numeric = build_features(test, TEST_ROOT, test_series, False)
print(train_numeric.shape, test_numeric.shape)

In [ ]:
vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=1)
text_train = vectorizer.fit_transform(train_text)
text_test = vectorizer.transform(test_text)
scaler = StandardScaler()
numeric_train = scaler.fit_transform(train_numeric)
numeric_test = scaler.transform(test_numeric)
X_train = hstack([text_train, numeric_train]).tocsr()
X_test = hstack([text_test, numeric_test]).tocsr()
print("Design matrices:", X_train.shape, X_test.shape)

In [ ]:
predictions = np.zeros((len(test), len(TARGETS)), dtype=np.float32)
train_predictions = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
models = {}
for j, target in enumerate(TARGETS):
    y = train[target].fillna(0).astype(int).to_numpy()
    if np.unique(y).size < 2:
        predictions[:, j] = y.mean()
        train_predictions[:, j] = y.mean()
        continue
    model = LogisticRegression(max_iter=300, class_weight="balanced", random_state=SEED)
    model.fit(X_train, y)
    predictions[:, j] = model.predict_proba(X_test)[:, 1]
    train_predictions[:, j] = model.predict_proba(X_train)[:, 1]
    models[target] = model
auc = [roc_auc_score(train[t], train_predictions[:, j]) for j, t in enumerate(TARGETS) if train[t].nunique() > 1]
print(f"Training macro ROC-AUC: {np.mean(auc):.5f}")

In [ ]:
submission = test[[ID]].copy()
submission[TARGETS] = np.clip(predictions, 0, 1)
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
assert list(submission.columns) == [ID] + TARGETS
assert submission[TARGETS].notna().all().all()
joblib.dump({"targets": TARGETS, "vectorizer": vectorizer, "scaler": scaler, "models": models}, "/kaggle/working/rsna_knee_baseline.joblib")
print("Submission saved:", submission_path)
display(submission.head())

## Entregable

Descarga `submission.csv` desde el panel de archivos de Kaggle y utiliza **Submit to Competition**. Conserva tambi?n `rsna_knee_baseline.joblib` y esta notebook como evidencia reproducible.